# Budgetwise ML Model Training (Jupyter)

Use this notebook to train/export `model.pkl` for the ML service.

Workflow:
1. Prepare `ml-service/training_expenses.json` (list of `{ title, category, amount }`).
2. Run all cells.
3. Deploy service with `ML_MODEL_PATH=model.pkl` (or custom path).

In [ ]:
from pathlib import Path
import json
from collections import Counter

from category_predictor import CategoryPredictor

In [ ]:
cwd = Path.cwd().resolve()
if (cwd / "category_predictor.py").exists():
    ML_DIR = cwd
elif (cwd / "ml-service" / "category_predictor.py").exists():
    ML_DIR = cwd / "ml-service"
else:
    raise RuntimeError("Run notebook from repository root or ml-service directory.")

MODEL_OUTPUT_PATH = ML_DIR / "model.pkl"
TRAINING_DATA_PATH = ML_DIR / "training_expenses.json"
METRICS_OUTPUT_PATH = ML_DIR / "model_metrics.json"

print("ML directory:", ML_DIR)
print("Model output:", MODEL_OUTPUT_PATH)
print("Training data:", TRAINING_DATA_PATH)

## Training Data

Create `training_expenses.json` in `ml-service` using `training_expenses.sample.json` as template.
Minimum 10 valid rows are required.

In [ ]:
def load_training_expenses(path: Path):
    if not path.exists():
        raise FileNotFoundError(f"Missing training file: {path}")

    with path.open("r", encoding="utf-8") as handle:
        payload = json.load(handle)

    if not isinstance(payload, list):
        raise ValueError("training_expenses.json must contain a JSON array")

    cleaned = []
    for row in payload:
        title = str((row or {}).get("title", "")).strip()
        category = str((row or {}).get("category", "")).strip().lower()
        amount = float((row or {}).get("amount", 0) or 0)

        if title and category:
            cleaned.append({
                "title": title,
                "category": category,
                "amount": amount,
            })

    if len(cleaned) < 10:
        raise ValueError(f"Need at least 10 valid rows, found {len(cleaned)}")

    return cleaned

expenses = load_training_expenses(TRAINING_DATA_PATH)
print(f"Loaded {len(expenses)} rows")
Counter(row["category"] for row in expenses)

In [ ]:
predictor = CategoryPredictor(model_path=str(MODEL_OUTPUT_PATH), user_id="notebook")
accuracy = predictor.train(expenses)

metrics = {
    "samples": len(expenses),
    "accuracy": float(accuracy),
    "categories": sorted({row["category"] for row in expenses}),
    "model_path": str(MODEL_OUTPUT_PATH),
}

with METRICS_OUTPUT_PATH.open("w", encoding="utf-8") as handle:
    json.dump(metrics, handle, indent=2)

print("Saved model to:", MODEL_OUTPUT_PATH)
print("Saved metrics to:", METRICS_OUTPUT_PATH)
metrics

In [ ]:
test_titles = [
    "uber office ride",
    "grocery store",
    "electricity payment",
    "movie ticket",
]

for title in test_titles:
    predicted_category, confidence = predictor.predict(title, 0)
    print(f"{title:20s} -> {predicted_category:20s} ({confidence:.2f})")

## Integrate With Deployed Service

Set ML service environment variable:

- `ML_MODEL_PATH=model.pkl` for shared model
- `ML_MODEL_PATH=models/model_{user_id}.pkl` for per-user models

Then restart ML service so it loads the exported artifact.